We have a model trained to look at the center on a large noisy dataset.  
Now train the model on simpler transforms in phases

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.neg_mask.model.datasets.blur_pad_dl import random_tfm, BlurPadDataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, mkdir, DiskImage, DiskBooleanMask
from pytorch_grad_cam import (
    GradCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
from mtrain.neg_mask.model.show import (
    get_preds_for_ds,
    show_classification_report,
    show_confusion_matrix,
    show_confusion_matrix_using_preds,
)
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox
from functools import partial
from sklearn.model_selection import train_test_split
from fastai.basics import DataLoaders, default_device
from mtrain.denorm import denormalize_imagenet, denormalize_4chan_imagenet
from mtrain.utils import show, it_chain
from fastai.callback.all import ProgressCallback
from fastai.basics import F1Score, Precision, Recall, CrossEntropyLossFlat
from fastai.vision.all import vision_learner, xresnet18, xresnet34

In [ ]:
! ls {FOVEATED_PATH}/train | head

In [ ]:
FOVEATED_PATH = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated")
DS_PATH = FOVEATED_PATH

In [ ]:
from fastai.callback.tracker import SaveModelCallback, Recorder

CLS_WEIGHT = torch.tensor([1.0, 3.0]).float().to("mps")


def get_learner(dls):
    learn = vision_learner(
        dls,
        xresnet34,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=3,
        pretrained=True,
    )
    learn = learn.remove_cb(ProgressCallback)


    learn.remove_cb(Recorder)
    learn.remove_cb(SaveModelCallback)
    learn.add_cb(Recorder())
    learn.add_cbs([SaveModelCallback(monitor="f1_score", fname="best")])
    return learn


def get_denormalized(tens):
    image, mask = None, None
    image = denormalize_imagenet(tens)
    image = image.permute([1, 2, 0]).numpy()
    if mask is not None:
        mask = mask.numpy()
    return image, mask


def show_gradcam_for_image(
    learn, input_tensor, target_label_idx=None, layer_name="0.7.1.conv1"
):
    target_layers = [learn.model.get_submodule(layer_name)]
    img_arr, _ = get_denormalized(input_tensor[0])

    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs

        return visualization, img_arr, model_outputs


def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)
    return probs, targs, decoded, losses

In [ ]:

def get_dls(num_samples, tfm, crop_size, tfms_crop_size, bbox_pad):
    image_paths = list((DS_PATH / "train").glob("*.jpg"))[:num_samples]
    stratify = [BlurPadDataset.label_func(p) for p in image_paths]
    train_paths, valid_paths = train_test_split(
        image_paths, test_size=0.2, stratify=stratify, random_state=42
    )

    train_ds = BlurPadDataset(train_paths, DS_PATH / "masks", crop_size, False, tfm, bbox_pad, tfms_crop_size=tfms_crop_size)
    valid_ds = BlurPadDataset(valid_paths, DS_PATH / "masks", crop_size, True, tfm, bbox_pad, tfms_crop_size=tfms_crop_size)
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=16,
        # pin_memory=True,
        persistent_workers=True,
    )  # don't respawn workers each epoch)
    return dls


def vis_sample(dls, idx):
    ds = dls.train_ds
    tens, targ = ds[idx]
    print("target", targ)
    print("shape", tens.shape)
    img, _ = get_denormalized(tens)
    plt.imshow(img, cmap="gray")
    plt.show()

# test dls

In [ ]:
# to counter the problem of the model focusing on texture/noise
# we decrease the probability of adding noise with each sweep while maintaining accuracy
# the next step is to remove overwrite noise
# then next is decreasing the add noise frequency
# first i would need to seee the performance of the model
#  on different types of aux transforms (step down? gaussian? blur?)
# our final model has no noise, and one kind of step down function
# we need to test it on all transforms and find the winner
# for each we do successive training by decreasing the add_noise chance parameter
def blur_tfm(
    cropped_image, mask, inner_bbox, add_noise_chance, blur_kernel_sz, blur_sigma
):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_blur(blur_kernel_sz, blur_sigma)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask


def step_down_tfm(cropped_image, mask, inner_bbox, add_noise_chance, ratio):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down(ratio)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask


def step_down_gauss_tfm(cropped_image, mask, inner_bbox, add_noise_chance, min_value):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(min_value)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask

# initialise learner

In [ ]:
def get_initialised_learner():
    dls = get_dls(100, random_tfm, 224, 224, 10)
    learner = get_learner(dls)
    # SUCC_UNBLUR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur")
    # state_dict = torch.load(SUCC_UNBLUR / "tfm-random_samples-5000_arch-xresnet18_iter-30.pth")
    # learner.model.load_state_dict(state_dict)
    return learner

# train blur tfm model

In [ ]:
btfm_60 = partial(btfm, add_noise_chance=0.6)
btfm_30 = partial(btfm, add_noise_chance=0.3)
btfm_15 = partial(btfm, add_noise_chance=0.15)
btfm_5 = partial(btfm, add_noise_chance=0.05)
btfm_0 = partial(btfm, add_noise_chance=-1)

dls = get_dls(100, btfm_0)

In [ ]:
vis_sample(dls, 7)

In [ ]:
blur_learner = get_initialised_learner()

In [ ]:
dls = get_dls(5000, btfm_60)
blur_learner.dls = dls

In [ ]:
blur_learner.fine_tune(1)
blur_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(5000, btfm_30)
blur_learner.dls = dls
blur_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(5000, btfm_15)
blur_learner.dls = dls
blur_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(5000, btfm_0)
blur_learner.dls = dls
blur_learner.fit_one_cycle(10)

In [ ]:
SUCC_UNBLUR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur")
torch.save(blur_learner.model.state_dict(), SUCC_UNBLUR / "tfm-blur_samples-5000-xresnet18_iter-20.pth")

In [ ]:
blur_learner = None

# train step_down model

In [ ]:
st_ed_tfm = partial(step_down_tfm, ratio=0.5)

In [ ]:
st_ed_learner = get_initialised_learner()

In [ ]:
st_ed_tfm_50 = partial(st_ed_tfm, add_noise_chance=0.5)
st_ed_tfm_15 = partial(st_ed_tfm, add_noise_chance=0.15)
st_ed_tfm_0 = partial(st_ed_tfm, add_noise_chance=-1)

In [ ]:
dls = get_dls(500, random_tfm, 224, 224, 3)
# st_ed_learner.dls = dls
# st_ed_learner.fine_tune(1)
# st_ed_learner.fit_one_cycle(5)

In [ ]:
vis_sample(dls, 20)

In [ ]:
st_ed_learner.cbs

In [ ]:
dls = get_dls(200, random_tfm, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(20)

In [ ]:
dls = get_dls(500, random_tfm, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(2000, random_tfm, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(5000, random_tfm, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(5000, st_ed_tfm_50, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(20000, st_ed_tfm_50, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(20000, st_ed_tfm_15, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(20000, st_ed_tfm_0, 224, 224, 3)
# dls = get_dls(2000, sta_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(15)

In [ ]:
torch.save(st_ed_learner.model.state_dict(), "./xresnet34-foveated.pt")

In [ ]:
viz, img, _ = show_gradcam_for_image(st_ed_learner, dls.valid_ds[4][0].unsqueeze(0), 1, "0.7.1.convpath.1.0")
show([img, viz])
# plt.imshow(viz)

In [ ]:
dls = get_dls(20000, st_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.freeze()
st_ed_learner.fit_one_cycle(3)

In [ ]:
st_ed_learner.freeze()
st_ed_learner.fit_one_cycle(5)

In [ ]:
dls = get_dls(20000, st_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(5)

In [ ]:
vis_sample(st_ed_learner.dls, 1)

In [ ]:
res = show_reports(st_ed_learner)

In [ ]:
SUCC_UNBLUR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur")
torch.save(st_ed_learner.model.state_dict(), SUCC_UNBLUR / "tfm-stepdown_ratio-5_samples-all-xresnet18_iter-20.pth")

# Gaussian step down

In [ ]:
st_gs_tfm = partial(step_down_gauss_tfm, min_value=0.3)

In [ ]:
st_gs_learner = get_initialised_learner()

In [ ]:
st_gs_tfm_50 = partial(st_gs_tfm, add_noise_chance=0.5)
st_gs_tfm_15 = partial(st_gs_tfm, add_noise_chance=0.15)
st_gs_tfm_0 = partial(st_gs_tfm, add_noise_chance=-1)

In [ ]:
dls = get_dls(5000, st_gs_tfm_50)
st_gs_learner.dls = dls
st_gs_learner.fine_tune(1)
st_gs_learner.fit_one_cycle(5)

In [ ]:
dls = get_dls(5000, st_gs_tfm_15)
st_gs_learner.dls = dls
st_gs_learner.fit_one_cycle(5)

In [ ]:
dls = get_dls(5000, st_gs_tfm_0)
st_gs_learner.dls = dls
st_gs_learner.fit_one_cycle(5)

In [ ]:
st_gs_learner.loss_func

In [ ]:
dls = get_dls(20000, st_gs_tfm_0)
st_gs_learner.dls = dls
st_gs_learner.fit_one_cycle(5)

In [ ]:
SUCC_UNBLUR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur")
torch.save(st_gs_learner.model.state_dict(), SUCC_UNBLUR / "tfm-gaussstepdown_min-3_samples-all-xresnet18_iter-20.pth")

In [ ]:
state_dict = torch.load(SUCC_UNBLUR / "tfm-gaussstepdown_min-3_samples-all-xresnet18_iter-20.pth")
st_gs_learner.model.load_state_dict(state_dict)

In [ ]:
res = show_reports(st_gs_learner)

In [ ]:
from fastai.basics import FocalLossFlat
fl_weight = torch.tensor([1.0, 2.0]).to(default_device())
st_gs_learner.loss_func = FocalLossFlat(weight=fl_weight, gamma=2)

In [ ]:
st_gs_learner.freeze()
st_gs_learner.fit_one_cycle(3)

In [ ]:
st_gs_learner.loss_func = CrossEntropyLossFlat(weight=fl_weight)

In [ ]:
st_gs_learner.lr_find()

In [ ]:
st_gs_learner.fit_one_cycle(10, slice(1e-4, 1e-3))

In [ ]:
SUCC_UNBLUR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur")
torch.save(st_gs_learner.model.state_dict(), SUCC_UNBLUR / "tfm-gaussstepdown_min-3_samples-all-xresnet18_iter-50.pth")

In [ ]:
st_gs_learner.lr_find()

In [ ]:
st_gs_learner.fit_one_cycle(5, slice(1e-5, 1e-4))

In [ ]:
res = show_reports(st_gs_learner)
probs, targs, decoded, losses = res

In [ ]:
res = show_reports(st_gs_learner)
probs, targs, decoded, losses = res

In [ ]:
res = show_reports(st_gs_learner)
probs, targs, decoded, losses = res

In [ ]:
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]


In [ ]:
idx = top_loss_idxs[6]
viz, img, mo = show_gradcam_for_image(st_ed_learner,  st_gs_learner.dls.valid_ds[idx][0].unsqueeze(0), 1, "0.7.1.convpath.1.0")
print(mo)
show([viz, img])


In [ ]:
dls = get_dls(20000, st_ed_tfm_15)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(10)

In [ ]:
dls = get_dls(20000, st_ed_tfm_0)
st_ed_learner.dls = dls
st_ed_learner.fit_one_cycle(20)

In [ ]:
SUCC_UNBLUR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur")
torch.save(st_ed_learner.model.state_dict(), SUCC_UNBLUR / "tfm-stepdown_ratio-3_samples-all-xresnet18_iter-20.pth")

In [ ]:
res =show_reports(st_ed_learner)

In [ ]:
probs, targs, decoded, losses = res
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]


In [ ]:
idx = top_loss_idxs[16]
viz, img = show_gradcam_for_image(st_ed_learner,  st_ed_learner.dls.valid_ds[idx][0].unsqueeze(0), 1, "0.7.1.convpath.1.0")
show([viz, img])


# train blur tfm model

In [ ]:
5000